# JWST image alignment with JHAT / Gaia

Relative and Gaia-based WCS alignment using `st123` (`jwst_phot`, `query_gaia`,
`calc_dispersion`, `align_jwst_image`, `expand_mask`, `add_bin_dq`).

CLI equivalent: `align --ref ... --image ... --base-dir ...`

Based on https://jhat.readthedocs.io/en/latest/examples/plot_b_nircam.html


In [ ]:
import sys
from pathlib import Path

# Resolve repo root whether cwd is repo, st123/, or st123/notebooks/
_here = Path.cwd().resolve()
ROOT = None
for candidate in [_here, *_here.parents]:
    if (candidate / 'pyproject.toml').is_file() and (candidate / 'st123').is_dir():
        ROOT = candidate
        break
if ROOT is None:
    ROOT = _here
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

from jhat import st_wcs_align

from st123 import (
    jwst_phot,
    query_gaia,
    calc_dispersion,
    align_jwst_image,
    add_bin_dq,
)


## Select reference and align images

Point `workdir` at a reduction directory that contains reference `*_i2d.fits`
and CAL frames to align.


In [ ]:
workdir = Path('jwstred_temp_gaia')

ref_candidates = (
    list(workdir.glob('*f277w*i2d.fits'))
    + list(workdir.glob('*f277w*_jhat_i2d.fits'))
    + list(workdir.glob('*i2d.fits'))
)
align_candidates = (
    list(workdir.glob('*nrcalong_cal.fits'))
    + list(workdir.glob('*nrcblong_cal.fits'))
    + list(workdir.glob('*_cal.fits'))
)

ref_image = str(ref_candidates[0]) if ref_candidates else None
align_image = str(align_candidates[0]) if align_candidates else None

print('workdir:', workdir.resolve())
print('ref_image:', ref_image)
print('align_image:', align_image)
if ref_image is None or align_image is None:
    raise FileNotFoundError(
        f'Need a reference i2d and a CAL frame under {workdir}. '
        'Update workdir or copy data into place before running alignment cells.'
    )

## Relative alignment

Build a photometric catalog from the reference image, then register the CAL frame with JHAT.


In [ ]:
refcat, ref_catname = jwst_phot(ref_image)
print('Reference catalog:', ref_catname, 'n=', len(refcat))
refcat[:5]

In [ ]:
wcs_align = st_wcs_align()
wcs_align.run_all(
    align_image,
    telescope='jwst',
    outsubdir=str(workdir),
    refcat_racol='ra',
    refcat_deccol='dec',
    refcat_magcol='mag',
    refcat_magerrcol='dmag',
    overwrite=True,
    d2d_max=1,
    showplots=2,
    refcatname=ref_catname,
    histocut_order='dxdy',
    sharpness_lim=(0.3, 0.9),
    roundness1_lim=(-0.7, 0.7),
    SNR_min=3,
    dmag_max=1.0,
)

## Align to Gaia

`align_jwst_image(..., gaia=True)` is the pipeline wrapper used by the scripts.


In [ ]:
aligned = align_jwst_image(
    align_image=align_image,
    outdir=str(workdir),
    gaia=True,
    verbose=True,
    plot=True,
)
aligned

In [ ]:
tb_gaia = query_gaia(ref_image, telescope='jwst', save_file=False)
mean_d, med_d, std_d = calc_dispersion(tb_gaia, ref_catname, plot=True)
print(f'dispersion (mas): mean={mean_d:.2f}, median={med_d:.2f}, std={std_d:.2f}')
mean_d, med_d, std_d

## DQ / breakup-star mask helpers

`add_bin_dq` writes a `*_masked.fits` product with an expanded BIN_DQ extension.


In [ ]:
cal_candidates = list(workdir.glob('*nrcblong_cal.fits')) + list(workdir.glob('*_cal.fits'))
cal = str(cal_candidates[0]) if cal_candidates else None
if cal is None:
    print('No CAL frame found for DQ masking.')
else:
    masked = add_bin_dq(cal)
    print('Wrote', masked)
    masked